# Image visualization using napari viewer

In [1]:
import napari
import pandas as pd
import tifffile
import os
import numpy as np

## Settings

In [6]:
root_path = "../data"
resolution = "preview_mode" # "cell_mode" or "preview_mode"

# choose img number if "standard" is selected above
img_nb = 13 # from 0-13

panel_file = "../data/cell_mode/panel.csv"
panel = pd.read_csv(panel_file)
print(len(panel.index), "channels in panel")

40 channels in panel


## Collect images and masks

In [10]:
if resolution == "preview_mode":
    img_file = os.path.join(root_path, resolution, "img/pixel_skip.tiff")
    img = tifffile.imread(img_file)
    mask_file = os.path.join(root_path, resolution, "masks/pixel_skip_mask.tiff")
    mask = tifffile.imread(mask_file).astype(np.uint16)
    img_dict = {resolution: img}
    masks_dict = {"tissue_mask": mask}
else:
    imgs = ["raw","steinbock","dimr"]
    img_dict = {}
    file_name = os.listdir(os.path.join(root_path, resolution, f"img_{imgs[0]}"))[img_nb]
    print(f"Analysing image: {file_name}")
    for n, img in enumerate(imgs):
        img_path = os.path.join(root_path, resolution, f"img_{img}", os.path.basename(file_name))
        img = tifffile.imread(img_path)
        img_dict[imgs[n]] = img
    masks = ["cellpose","instanseg","cellpose_sam"]
    masks_dict = {}
    for n, mask in enumerate(masks):
        mask_path = os.path.join(root_path, resolution, f"masks_{mask}", os.path.basename(file_name))
        mask = tifffile.imread(mask_path)
        masks_dict[masks[n]] = mask

## Visualize multiple images/masks as single, scrollable layers

In [4]:
def compare_hpfs_and_masks(img_dict, masks_dict):
    viewer = napari.Viewer()

    viewer.axes.visible = True
    viewer.dims.axis_labels = ("y", "x")
    
    viewer.scale_bar.visible = True
    viewer.scale_bar.unit = "um"

    for key, img in img_dict.items():
        viewer.add_image(img, channel_axis=None, name=key)

    for key, mask in masks_dict.items():
        mask_layer = viewer.add_labels(
            data=mask,
            name=key,
            blending="translucent",
            visible=False,
        )

In [5]:
compare_hpfs_and_masks(img_dict, masks_dict)

## Visualize single images/masks with each channel named as a layer

In [11]:
def single_layered_img(img, mask, panel):
    viewer = napari.Viewer()

    viewer.axes.visible = True
    viewer.dims.axis_labels = ("y", "x")
    
    viewer.scale_bar.visible = True
    viewer.scale_bar.unit = "um"
    
    img_layers = viewer.add_image(
        data=img,
        channel_axis=0,
        colormap="gray",
        name=panel.loc[panel["keep"] == 1, "name"],
        blending="additive",
        visible=False,
    )
    
    mask_layer = viewer.add_labels(
        data=mask,
        name="Mask",
        blending="translucent",
        visible=False,
    )

In [12]:
single_layered_img(img_dict["preview_mode"], masks_dict["tissue_mask"], panel)